In [32]:
# loading in libraries
import json
from pathlib import Path
from pprint import pprint
from langchain_community.document_loaders import JSONLoader
import os
import getpass
import ApiKeyData
from langchain_openai import ChatOpenAI
#from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.vectorstores import Chroma
import chromadb
import runhouse as rh
#from langchain_community.embeddings import SelfHostedHuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain.schema import Document

In [33]:
#setting up model
llm = ChatOpenAI(openai_api_key=ApiKeyData.LANGCHAIN_API_KEY, temperature=0.7)
#temperature helps with creativeness of the responses the smaller the temperature the more direct the response will be

# add extra_body if I want to add to the current json data

In [34]:
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large") # <- better data but costly to implement
#embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2") # <- cost effective option

# gpu = rh.cluster(name="rh-a10x", instance_type="A100:1") #creates a cluster that helps with high-performance computing and deep learning tasks - helps accelerate embedding task
# embeddings = SelfHostedHuggingFaceEmbeddings(model_id=model_id, hardware=gpu)


In [35]:
# loading in data
file_path = './winemag-data-130k-v2.json'
data = json.loads(Path(file_path).read_text())
# will print data

In [36]:
print(data[0])
print(len(data))

{'points': '87', 'title': 'Nicosia 2013 Vulkà Bianco  (Etna)', 'description': "Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.", 'taster_name': 'Kerin O’Keefe', 'taster_twitter_handle': '@kerinokeefe', 'price': None, 'designation': 'Vulkà Bianco', 'variety': 'White Blend', 'region_1': 'Etna', 'region_2': None, 'province': 'Sicily & Sardinia', 'country': 'Italy', 'winery': 'Nicosia'}
129971


In [59]:
# CREATING DOCUMENTS SO DATA CAN BE EMBEDDED
documents = []
metadatas = []
i = 0
for entry in data: 
    # Construct each document
    doc = Document(
        page_content=entry.get('description', 'variety'),  # or another field like 'title'
        #metadata=entry  # You can store all other fields in metadata for later use
    )
    documents.append(doc)
    metadatas.append(entry)
    i += 1
    if i == 1000:
        break

# Verify the number of documents created
documentString = [item.page_content for item in documents]

print(f"Number of documents created: {len(documents)}")

Number of documents created: 1000


In [38]:
# LOADING DATA
#NOT WORKING
# loader = JSONLoader(file_path="./winemag-data-130k-v2.json", jq_schema=".", text_content=False)
# documents = loader.load() #loads data into document objects

# jq_schema="." extracts data from the whole json file not just a single array
# text_content=False tells model that this is structured data not raw text

In [97]:
# EMBEDDING
texts = [doc.page_content for doc in documents] #extracts text for embedding

model = SentenceTransformer('all-MiniLM-L6-v2') #same model that Chromadb uses - intializing embedding model

document_embeddings = model.encode(texts) #converts text into numerical embeddings - converts each document into a vector representation (embedding)

# NOTE SINCE THERE ARE SO MANY DOCUMENTS CONSIDER SAVING EMBEDDINGS SO I DONT EVER HAVE TO RUN THIS AGAIN
# ALSO NOTE THAT YOU CAN RUN MULTIPLE DOCUMENTS AT THE SAME TIME WITH CERTAIN PROGRAMS 
# (ASYNCHRONOUS PROGRAMMING)

In [98]:
print(len(ids))

384


In [99]:
print(len(document_embeddings))

1000


In [100]:
print(metadatas[0])

{'points': '87', 'title': 'Nicosia 2013 Vulkà Bianco  (Etna)', 'description': "Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.", 'taster_name': 'Kerin O’Keefe', 'taster_twitter_handle': '@kerinokeefe', 'price': None, 'designation': 'Vulkà Bianco', 'variety': 'White Blend', 'region_1': 'Etna', 'region_2': None, 'province': 'Sicily & Sardinia', 'country': 'Italy', 'winery': 'Nicosia'}


In [101]:
# documentString = [item.page_content for item in documents]
# metadataString = [item.metadata for item in metadatas] <- ignore this for now

In [102]:
print(documents[0])
print(documentString[0]) #<- converted it so it would be usable with Chromadb
print(len(documentString))
print(len(ids))
#documentString = [item['page_content'] for item in documents]

page_content='Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.'
Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
1000
384


In [103]:
chroma_client.delete_collection(name="wine_data_collection")

In [104]:
# # VECTORIZING DATA

chroma_client = chromadb.Client() # inializes chromadb so we can connect our collection to the vector storage

collection = chroma_client.create_collection(name="wine_data_collection") #creating collection - this is essentially a vector db that can store embeddings, queries, and documents
# NOTE - The collection name must start and end with a lowercase letter
ids = [f"doc_{i}" for i in range(len(document_embeddings))]

collection.add(
    ids=ids,  #list of unqiue ids connected to documents
    embeddings=document_embeddings,  # List of embeddings
    #metadatas=None, #optional
    documents=documentString #optional
)

# retriever = db.as_retriever()

# If Chroma is passed a list of documents, it will automatically tokenize and embed them with the
#  collection's embedding function (the default will be used if none was supplied at collection creation)
# Chroma will also store the documents themselves.

In [105]:
print(collection.peek())

{'ids': ['doc_0', 'doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5', 'doc_6', 'doc_7', 'doc_8', 'doc_9'], 'embeddings': array([[ 0.03865863, -0.01439859,  0.06814581, ...,  0.0036253 ,
         0.01293009, -0.02271832],
       [ 0.01121124, -0.08460533, -0.0878832 , ..., -0.02558688,
         0.09891669, -0.06970023],
       [-0.00550225, -0.06958877,  0.03468077, ..., -0.02910002,
         0.07911143, -0.00786981],
       ...,
       [-0.00040771, -0.01770479, -0.01707942, ..., -0.01976915,
         0.02667214, -0.07023711],
       [-0.02644042, -0.04240281,  0.01873131, ...,  0.01169297,
         0.03819253, -0.06623403],
       [ 0.0029559 , -0.06621409,  0.02600192, ...,  0.0326559 ,
         0.01003386, -0.02346071]]), 'documents': ["Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.", "This is ripe and fruity, a wine that is smooth while still structured. Firm tanni

In [116]:
results = collection.query(
    query_texts=["tobacco wine?"], #"can you give me a smoky wine?"
    n_results=3,
    #include=["documents"]
)
results

{'ids': [['doc_633', 'doc_859', 'doc_440']],
 'embeddings': None,
 'documents': [['This full-bodied wine is rich without ever seeming overdone. Subtle smoke notes frame bold fruit—tangerine, pineapple and apricot—and gingery spice, while the finish is long, peppery and refreshing. Drink now–2020.',
   'This is a generous, leathery and tannic wine, full-bodied and made in a juicy, foreboding style. Black pepper, leather and cigar smoke provide a sultry, robust edge to the wine, which should cellar well, through 2020.',
   'This wine is a blend of fruit from five different vineyards scattered across the valley. The aromas of barrel spice, black cherry, dried leaves, herbs and flowers still seem quite locked up. The flavors are generous and full with palate coating milk chocolate and cherry flavors with a finish that lingers.']],
 'uris': None,
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.7315258383750916, 0.7588546276092529, 0.7802954912185669]],
 'included': [<In

In [29]:
# pprint(data)

In [31]:
print(len(data))

129971


In [7]:
# consider splitting document to help with processing speed - could split by , for each wine is its own section {}""

In [9]:
# add openai embeddings and vector store



In [23]:
#vectors connect data to 
# vector_store = InMemoryVectorStore(embeddings)

# index = VectorstoreIndexCreator(
#     embedding=embedding,
#     vectorstore_cls=DocArrayInMemorySearch
# ).from_loaders([data])

In [63]:
# results = vector_store.similarity_search(
#     "How many wines are from Napa?"
# )

# print(results[0])